In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# インポート
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

In [ ]:
# train読み込み
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
train.head()

In [ ]:
# idは不要なので消去
train = train.drop(columns=["id"])
train.head()

In [ ]:
# storeデータ読み込み
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
stores.head()

In [ ]:
# oilデータ読み込み
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
oil.head()

In [ ]:
# holidaysデータ読み込み、表示
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
holidays.head()

In [ ]:
# データ整形
# type
# 休日を１つにまとめる
holidays["is_holiday"] = holidays["type"].isin(["Holiday", "Additional", "Bridge"])
# イベントは個別の特徴
holidays["is_event"] = holidays["type"] == "Event"
# 休日でない場合(必要に応じて)
# holidays["is_workday"] = holidays["type"] == "Work Day"

# 表示
holidays.head()

In [ ]:
# typeは分割済みなので削除
holidays = holidays.drop(columns=["type"])
holidays.head()

In [ ]:
# transactionsデータ読み込み、表示
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')
transactions.head()

In [ ]:
# データを結合し、mergedに格納
merged_s = pd.merge(train, stores, on="store_nbr",how="left")
merged_so = pd.merge(merged_s, oil, on="date",how="left")
merged_soh = pd.merge(merged_so, holidays, on="date",how="left")
merged = pd.merge(merged_soh, transactions, on=["date","store_nbr"],how="left")

merged.head()

In [ ]:
# 欠損値を補う
# dcoilwticoの補完、前方補完(直前のデータで補完)、直前のデータがない場合は後方補完
merged["dcoilwtico"] = merged["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")
# transactions(取引数)がNaN(欠損)のときは取引数0であるため変換
merged["transactions"] = merged["transactions"].fillna(0)
# is_holiday,is_eventが欠損の場合は普通の日のためFalseに変換
merged[["is_holiday", "is_event"]] = merged[["is_holiday", "is_event"]].fillna(False)
# transferredが欠損のとき,False
merged["transferred"] = merged["transferred"].fillna(False)
# 表示
merged.head()

In [ ]:
# date分割
# dateをdatetime型に変換
merged["date"] = pd.to_datetime(merged["date"])

# 年、月、日をそれぞれの新しい列に分ける
merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["day"] = merged["date"].dt.day

# 曜日（0=月曜日）
merged["dayofweek"] = merged["date"].dt.dayofweek
# 休日(土日)かどうか
merged["is_weekend"] = merged["dayofweek"] >= 5
# 周番号(必要に応じて)
# merged["weekofyear"] = merged["date"].dt.isocalendar().week

# 表示
merged.head()

In [ ]:
# city,stateとlocale,locale_nameを用いて、店舗の地域がholiday,eventの対象か確認、対象でない場合Falseに変更
merged.loc[(merged["locale"] == "Local") & (merged["city"] != merged["locale_name"]),["is_holiday","is_event"]] = False
merged.loc[(merged["locale"] == "Regional") & (merged["state"] != merged["locale_name"]),["is_holiday","is_event"]] = False
# 表示
merged.head()

In [ ]:
# 不要になったlocate,locate_nameを削除
merged = merged.drop(columns=["locale","locale_name"])
# 表示
merged.head()

In [ ]:
# 使わないものを削除
merged = merged.drop(columns=["city","state","description","date"])
# 表示
merged.head()

In [ ]:
# ダミー変数化
training=pd.get_dummies(merged, columns=["store_nbr","family","type","cluster","dayofweek"], drop_first=True)
training.head()

In [ ]:
# 年、月、日のデータを標準化
from sklearn.preprocessing import StandardScaler
train_standard = StandardScaler()
train_copied = training.copy()
train_standard.fit(train_copied[['year', 'month','day']])
train_std = pd.DataFrame(train_standard.transform(train_copied[['year', 'month','day']]))
train_std.head()

In [ ]:
# 元の年、月、日のデータを削除し、標準化したデータを結合
train_std.columns = ['year_std', 'month_std', 'day_std']
training = training.drop(columns=['year', 'month', 'day'])
training = pd.concat([training, train_std], axis=1)
training.head()

In [ ]:
"""
# 要素確認
print(training.columns.tolist())

# 欠損の確認(Trueであれば欠損無し)
print(training.isnull().sum().sum() == 0)
print(training.isnull().sum()[training.isnull().sum() > 0])

# 特徴量のデータ型
print(training.dtypes.value_counts())
# 特定の型のカラムだけ表示
# print(training.select_dtypes(include=['float']).columns.tolist())
"""

In [ ]:
# test読み込み
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
test.head()
# testデータはidが必要

In [ ]:
# データを結合し、merged_testに格納
merged_ts = pd.merge(test, stores, on="store_nbr",how="left")
merged_tso = pd.merge(merged_ts, oil, on="date",how="left")
merged_tsoh = pd.merge(merged_tso, holidays, on="date",how="left")
merged_test = pd.merge(merged_tsoh, transactions, on=["date","store_nbr"],how="left")

merged_test.head()

In [ ]:
# 欠損値を補う
# dcoilwticoの補完、前方補完(直前のデータで補完)、直前のデータがない場合は後方補完
merged_test["dcoilwtico"] = merged_test["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")
# transactions(取引数)がNaN(欠損)のときは取引数0であるため変換
merged_test["transactions"] = merged_test["transactions"].fillna(0)
# is_holiday,is_eventが欠損の場合は普通の日のためFalseに変換
merged_test[["is_holiday", "is_event"]] = merged_test[["is_holiday", "is_event"]].fillna(False)
# transferredが欠損のとき,False
merged_test["transferred"] = merged_test["transferred"].fillna(False)
# 表示
merged_test.head()

In [ ]:
# date分割
# dateをdatetime型に変換
merged_test["date"] = pd.to_datetime(merged_test["date"])

# 年、月、日をそれぞれの新しい列に分ける
merged_test["year"] = merged_test["date"].dt.year
merged_test["month"] = merged_test["date"].dt.month
merged_test["day"] = merged_test["date"].dt.day

# 曜日（0=月曜日）
merged_test["dayofweek"] = merged_test["date"].dt.dayofweek
# 休日(土日)かどうか
merged_test["is_weekend"] = merged_test["dayofweek"] >= 5
# 周番号(必要に応じて)
# merged_test["weekofyear"] = merged_test["date"].dt.isocalendar().week

# 表示
merged_test.head()

In [ ]:
# city,stateとlocale,locale_nameを用いて、店舗の地域がholiday,eventの対象か確認、対象でない場合Falseに変更
merged_test.loc[(merged_test["locale"] == "Local") & (merged_test["city"] != merged_test["locale_name"]),["is_holiday","is_event"]] = False
merged_test.loc[(merged_test["locale"] == "Regional") & (merged_test["state"] != merged_test["locale_name"]),["is_holiday","is_event"]] = False
# 表示
merged_test.head()

In [ ]:
# 不要になったlocate,locate_nameを削除
merged_test = merged_test.drop(columns=["locale","locale_name"])
# 表示
merged_test.head()

In [ ]:
# 使わないものを削除
merged_test = merged_test.drop(columns=["city","state","description","date"])
# 表示
merged_test.head()

In [ ]:
# ダミー変数化
testing=pd.get_dummies(merged_test, columns=["store_nbr","family","type","cluster","dayofweek"], drop_first=True)
testing.head()

In [ ]:
# 年、月、日のデータを標準化
from sklearn.preprocessing import StandardScaler
test_standard = StandardScaler()
test_copied = testing.copy()
test_standard.fit(test_copied[['year', 'month','day']])
test_std = pd.DataFrame(test_standard.transform(test_copied[['year', 'month','day']]))
test_std.head()

In [ ]:
# 元の年、月、日のデータを削除し、標準化したデータを結合
test_std.columns = ['year_std', 'month_std', 'day_std']
testing = testing.drop(columns=['year', 'month', 'day'])
testing = pd.concat([testing, test_std], axis=1)
testing.head()

In [ ]:
"""
# 要素確認
print(testing.columns.tolist())

# 欠損の確認(Trueであれば欠損無し)
print(testing.isnull().sum().sum() == 0)
print(testing.isnull().sum()[testing.isnull().sum() > 0])

# 特徴量のデータ型
print(testing.dtypes.value_counts())
# 特定の型のカラムだけ表示
# print(testing.select_dtypes(include=['float']).columns.tolist())
"""

In [ ]:
# 学習、予測（モデル：LightGBM（決定木の一種、決定木を複数回連続して行う））
# 特徴量と目的変数を分離
X = training.drop(columns=["sales"])
y = training["sales"]

In [ ]:
# 特徴量名に特殊文字がないように変換
X.columns = X.columns.str.replace('[^A-Za-z0-9_]', '_', regex=True)
testing.columns = testing.columns.str.replace('[^A-Za-z0-9_]', '_', regex=True)

In [ ]:
# 特徴量名を保存しておく（あとで testing に使う）
feature_names = X.columns

In [ ]:
# 2. データ分割（学習 / 検証）
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
# LightGBMデータセット作成
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)

In [ ]:
# パラメータ設定
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

In [ ]:
# 学習
callbacks = [lgb.early_stopping(stopping_rounds=50)]
model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'valid'],
    num_boost_round=1000,
    callbacks=callbacks
)

In [ ]:
# 検証データで RMSLE を評価
y_val_pred = model.predict(X_val)
y_val_pred = np.maximum(0, y_val_pred)  # 負の予測を0に修正
rmsle = np.sqrt(mean_squared_log_error(y_val, y_val_pred))
print(f"Validation RMSLE: {rmsle:.4f}")

In [ ]:
# テストデータで予測
X_test = testing[feature_names]
y_test_pred = model.predict(X_test)
y_test_pred = np.maximum(0, y_test_pred)  # RMSLE対応で負値を0に補正

In [ ]:
# 8. 提出用ファイル作成（例：'id' 列 + 'sales' 列）
submission = pd.DataFrame({
    "id": testing["id"],
    "sales": y_test_pred
})
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv を保存しました")